[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kadimetla/AIML_Learning_Gang/blob/main/statistics/regression_methods/01_linear_regression/Linear_Regression.ipynb)  [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/kadimetla/AIML_Learning_Gang/blob/main/statistics/regression_methods/01_linear_regression/Linear_Regression.ipynb)

# Notebook 01 — Linear Regression
## Predicting a Continuous Number

**Dataset**: California Housing (20,640 census districts)
**Question**: *What is the median house value in this district?*
**Type**: Supervised Regression — output is a continuous number

---

## Section 1 — What Is Linear Regression?

### In plain English

Imagine you are a property assessor estimating house values across California.
You notice that districts with higher incomes, newer houses, and coastal locations
tend to have higher values. Linear regression finds the best formula to combine all
these clues into a single predicted number:

```
predicted_value = w1×MedInc + w2×HouseAge + w3×AveRooms + ... + bias
```

Each feature gets a **weight** — how much it shifts the prediction.

| Weight | Meaning |
|---|---|
| Large positive | Higher feature value → higher house value |
| Large negative | Higher feature value → lower house value |
| Near zero | Feature barely matters |

> This is the same model as `learning_methods/01`, now applied to a real regression problem.

## Section 2 — How Does It Learn?

Three steps, repeated thousands of times:

1. **Predict** — multiply each feature by its weight and sum
2. **Measure error** — Mean Squared Error: `MSE = average of (predicted − actual)²`
3. **Update weights** — gradient descent: take a tiny step in the direction that reduces MSE

The weights the model settles on ARE the rules it has learned.
A large positive weight on `MedInc` means: higher income districts have higher house values.

## Section 3 — What Does the Data Need?

### Gradient-based models need scaling

All three (Linear, Ridge, Lasso) use gradient descent — the same scaling argument applies:
features on different scales cause unequal gradient steps.

| Feature | Description | Transform |
|---|---|---|
| `MedInc` | Median income (in $10k) | StandardScaler |
| `HouseAge` | Median house age (years) | StandardScaler |
| `AveRooms` | Average rooms per household | StandardScaler + clip outliers |
| `AveBedrms` | Average bedrooms per household | StandardScaler + clip outliers |
| `Population` | Block group population | StandardScaler |
| `AveOccup` | Average household occupancy | StandardScaler + clip outliers |
| `Latitude` | Geographic latitude | StandardScaler |
| `Longitude` | Geographic longitude | StandardScaler |

> **Split first, then scale** — fit the scaler on training data only to avoid data leakage.

In [ ]:
from sklearn.datasets import fetch_california_housing
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['MedHouseVal'] = housing.target
for col in ['AveRooms', 'AveBedrms', 'AveOccup']:
    df[col] = df[col].clip(upper=df[col].quantile(0.99))

FEATURES = housing.feature_names.tolist()
TARGET   = 'MedHouseVal'

print(f"California Housing: {df.shape[0]} districts, {len(FEATURES)} features")
print(f"Target (MedHouseVal): median house value in $100k, range {df[TARGET].min():.2f}–{df[TARGET].max():.2f}")
df.head(6)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X_raw = df[FEATURES].values
y     = df[TARGET].values

X_tr_raw, X_te_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_tr_raw)
X_test  = scaler.transform(X_te_raw)

stats = pd.DataFrame({
    'Feature'     : FEATURES,
    'Raw mean'    : X_tr_raw.mean(axis=0).round(3),
    'Raw std'     : X_tr_raw.std(axis=0).round(3),
    'Scaled mean' : X_train.mean(axis=0).round(4),
    'Scaled std'  : X_train.std(axis=0).round(4),
})
print(f"Train: {len(X_train)} districts  |  Test: {len(X_test)} districts")
print()
print(stats.to_string(index=False))

## Section 4 — Train the Model and Read the Rules

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)
print(f"RMSE : {rmse:.4f}  (×$100k, so off by ~${rmse*100:.0f}k on average)")
print(f"R²   : {r2:.4f}  ({r2*100:.1f}% of house value variation explained)")
print()
coef_df = pd.DataFrame({'Feature': FEATURES, 'Weight': model.coef_.round(4)})
coef_df = coef_df.reindex(coef_df['Weight'].abs().sort_values(ascending=False).index)
print(f"Intercept: {model.intercept_:.4f}")
print()
print("Weights learned (sorted by magnitude):")
print(coef_df.to_string(index=False))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.scatter(y_test, y_pred, alpha=0.2, s=10, color='steelblue')
lim = max(y_test.max(), y_pred.max()) * 1.02
ax1.plot([0, lim], [0, lim], 'r--', lw=1.5, label='Perfect prediction')
ax1.set_xlabel('Actual value (×$100k)', fontsize=11)
ax1.set_ylabel('Predicted value (×$100k)', fontsize=11)
ax1.set_title(f'Actual vs Predicted\nRMSE={rmse:.3f}  R²={r2:.3f}', fontsize=12)
ax1.legend()

colors = ['#26A69A' if w >= 0 else '#EF5350' for w in coef_df['Weight']]
ax2.barh(coef_df['Feature'], coef_df['Weight'], color=colors)
ax2.axvline(0, color='black', lw=0.8)
ax2.set_title('Feature Weights\n(green = raises value  red = lowers value)', fontsize=12)
ax2.set_xlabel('Weight', fontsize=11)
plt.tight_layout()
plt.show()